In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter

ROOT_DIR = Path("..")

GT_PATH = ROOT_DIR / "training_datasets" / "train_ground_truth.tsv"
CANDIDATE_PATH = ROOT_DIR / "candidate_pairs.tsv"

CHUNK_SIZE = 500_000

print("Ground truth:", GT_PATH)
print("Candidates:  ", CANDIDATE_PATH)
print("GT exists:  ", GT_PATH.exists())
print("Candidates exists:", CANDIDATE_PATH.exists())

Ground truth: ../training_datasets/train_ground_truth.tsv
Candidates:   ../candidate_pairs.tsv
GT exists:   True
Candidates exists: False


In [4]:
gt_sample = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
    nrows=5,
    keep_default_na=False
)

gt_sample

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [5]:
def parse_entity_id(entity_id):
    """
    Convert:
        S1-123      -> (1, 123)
        S2-456      -> (2, 456)
        S3-789      -> (3, 789)
    """
    source = entity_id[:2]
    number = int(entity_id[3:])

    if source == "S1":
        source_code = 1
    elif source == "S2":
        source_code = 2
    elif source == "S3":
        source_code = 3
    else:
        raise ValueError(f"Unexpected entity ID: {entity_id}")

    return source_code, number

In [6]:
def make_pair_key(s1_id, source_id):
    """
    Encode an S1 -> S2/S3 pair into one uint64.

    Upper 32 bits  = S1 numeric ID
    Lower 32 bits  = source numeric ID

    The S2/S3 source prefix is also encoded into the lower part
    so S2-123 and S3-123 remain different entities.
    """

    s1_num = np.uint64(int(s1_id[3:]))

    source_code = 2 if source_id[:2] == "S2" else 3
    source_num = np.uint64(int(source_id[3:]))

    # Encode source into the upper bits of the lower 32 bits.
    source_key = (np.uint64(source_code) << np.uint64(29)) | source_num

    return (s1_num << np.uint64(32)) | source_key

In [7]:
true_key_chunks = []

total_gt_rows = 0
total_true_pairs = 0

for chunk in pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    keys = []

    for row in chunk.itertuples(index=False):
        s1_id = row.source1_entity_id
        matched_ids = row.matched_entity_ids

        if not matched_ids:
            continue

        for source_id in matched_ids.split(","):
            if source_id:
                keys.append(
                    make_pair_key(s1_id, source_id)
                )

    if keys:
        true_key_chunks.append(
            np.asarray(keys, dtype=np.uint64)
        )

        total_true_pairs += len(keys)

    total_gt_rows += len(chunk)

    print(
        f"GT rows: {total_gt_rows:,} | "
        f"true pairs: {total_true_pairs:,}"
    )

true_pair_keys = np.concatenate(true_key_chunks)

# Sort once so membership checks can use np.searchsorted
true_pair_keys.sort()

print("\nDone.")
print(f"Ground-truth S1 rows: {total_gt_rows:,}")
print(f"True pairs:           {len(true_pair_keys):,}")
print(f"Expected:             7,638,365")

GT rows: 100,000 | true pairs: 345,997
GT rows: 200,000 | true pairs: 693,069
GT rows: 300,000 | true pairs: 1,038,755
GT rows: 400,000 | true pairs: 1,384,645
GT rows: 500,000 | true pairs: 1,730,506
GT rows: 600,000 | true pairs: 2,076,133
GT rows: 700,000 | true pairs: 2,422,079
GT rows: 800,000 | true pairs: 2,768,341
GT rows: 900,000 | true pairs: 3,114,006
GT rows: 1,000,000 | true pairs: 3,460,754
GT rows: 1,100,000 | true pairs: 3,806,818
GT rows: 1,200,000 | true pairs: 4,153,634
GT rows: 1,300,000 | true pairs: 4,498,735
GT rows: 1,400,000 | true pairs: 4,844,772
GT rows: 1,500,000 | true pairs: 5,190,860
GT rows: 1,600,000 | true pairs: 5,537,259
GT rows: 1,700,000 | true pairs: 5,884,185
GT rows: 1,800,000 | true pairs: 6,230,774
GT rows: 1,900,000 | true pairs: 6,576,314
GT rows: 2,000,000 | true pairs: 6,921,867
GT rows: 2,100,000 | true pairs: 7,268,445
GT rows: 2,200,000 | true pairs: 7,614,812
GT rows: 2,206,821 | true pairs: 7,638,365

Done.
Ground-truth S1 rows: 2,20

In [8]:
unique_true_pairs = np.unique(true_pair_keys)

print("True pairs:        ", len(true_pair_keys))
print("Unique true pairs: ", len(unique_true_pairs))
print("Duplicates:        ", len(true_pair_keys) - len(unique_true_pairs))

True pairs:         7638365
Unique true pairs:  7638365
Duplicates:         0


In [9]:
def make_candidate_keys(s1_ids, source_ids):
    """
    Vectorized conversion of candidate S1/source IDs
    into uint64 pair keys.
    """

    s1_nums = s1_ids.str[3:].astype(np.uint64)

    source_codes = np.where(
        source_ids.str[:2].eq("S2"),
        np.uint64(2),
        np.uint64(3)
    )

    source_nums = source_ids.str[3:].astype(np.uint64)

    source_keys = (
        (source_codes << np.uint64(29))
        | source_nums
    )

    return (
        (s1_nums << np.uint64(32))
        | source_keys
    )

In [12]:
# ============================================================
# CELL 8 — BLOCKING RECALL VALIDATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Candidate pairs path
# ------------------------------------------------------------

CANDIDATE_PATH = Path("../candidate_data/candidate_pairs.tsv")

print("=" * 60)
print("Candidate file")
print("=" * 60)

print(f"Path:   {CANDIDATE_PATH.resolve()}")
print(f"Exists: {CANDIDATE_PATH.exists()}")

if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(
        f"candidate_pairs.tsv not found at:\n"
        f"{CANDIDATE_PATH.resolve()}"
    )

print(
    f"Size:   "
    f"{CANDIDATE_PATH.stat().st_size / (1024**3):.2f} GB"
)


# ------------------------------------------------------------
# 2. Validate ground-truth variables
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Ground truth")
print("=" * 60)

print(f"Unique true pairs: {len(true_pair_keys):,}")


# ------------------------------------------------------------
# 3. Scan candidate pairs
# ------------------------------------------------------------

recovered_true_pairs = 0
candidate_rows = 0

recovered_s2 = 0
recovered_s3 = 0

for chunk in pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):

    candidate_rows += len(chunk)

    # --------------------------------------------------------
    # Create comparable pair keys
    # --------------------------------------------------------

    candidate_keys = make_candidate_keys(
        chunk["s1_id"],
        chunk["source_id"]
    )

    # --------------------------------------------------------
    # Find candidate keys inside ground truth
    # --------------------------------------------------------

    positions = np.searchsorted(
        true_pair_keys,
        candidate_keys
    )

    valid = positions < len(true_pair_keys)

    matched = np.zeros(
        len(candidate_keys),
        dtype=bool
    )

    matched[valid] = (
        true_pair_keys[positions[valid]]
        == candidate_keys[valid]
    )

    # --------------------------------------------------------
    # Count recovered true pairs
    # --------------------------------------------------------

    recovered_true_pairs += int(
        matched.sum()
    )

    # --------------------------------------------------------
    # Count recovered pairs by source
    # --------------------------------------------------------

    if matched.any():

        matched_sources = chunk.loc[
            matched,
            "source"
        ]

        recovered_s2 += int(
            (matched_sources == "S2").sum()
        )

        recovered_s3 += int(
            (matched_sources == "S3").sum()
        )

    print(
        f"Candidates processed: "
        f"{candidate_rows:,} | "
        f"True pairs recovered: "
        f"{recovered_true_pairs:,}"
    )


# ------------------------------------------------------------
# 4. Final blocking recall
# ------------------------------------------------------------

total_true_pairs = len(true_pair_keys)

missed_true_pairs = (
    total_true_pairs
    - recovered_true_pairs
)

overall_recall = (
    recovered_true_pairs / total_true_pairs
)


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BLOCKING RECALL")
print("=" * 60)

print(
    f"Candidate pairs:        "
    f"{candidate_rows:,}"
)

print(
    f"True pairs:             "
    f"{total_true_pairs:,}"
)

print(
    f"Recovered true pairs:   "
    f"{recovered_true_pairs:,}"
)

print(
    f"Missed true pairs:      "
    f"{missed_true_pairs:,}"
)

print(
    f"Blocking recall:        "
    f"{overall_recall:.4%}"
)

print("\n" + "=" * 60)
print("RECOVERED TRUE PAIRS BY SOURCE")
print("=" * 60)

print(f"S2 recovered:           {recovered_s2:,}")
print(f"S3 recovered:           {recovered_s3:,}")

Candidate file
Path:   /Users/sunny/Documents/Codes/Amazon ML 2026/candidate_data/candidate_pairs.tsv
Exists: True
Size:   2.55 GB

Ground truth
Unique true pairs: 7,638,365
Candidates processed: 500,000 | True pairs recovered: 24,795
Candidates processed: 1,000,000 | True pairs recovered: 49,202
Candidates processed: 1,500,000 | True pairs recovered: 74,160
Candidates processed: 2,000,000 | True pairs recovered: 98,492
Candidates processed: 2,500,000 | True pairs recovered: 123,685
Candidates processed: 3,000,000 | True pairs recovered: 148,686
Candidates processed: 3,500,000 | True pairs recovered: 173,320
Candidates processed: 4,000,000 | True pairs recovered: 198,351
Candidates processed: 4,500,000 | True pairs recovered: 222,636
Candidates processed: 5,000,000 | True pairs recovered: 247,435
Candidates processed: 5,500,000 | True pairs recovered: 271,902
Candidates processed: 6,000,000 | True pairs recovered: 296,527
Candidates processed: 6,500,000 | True pairs recovered: 321,317
